# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [3]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3']
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
728,17,36,"(0.0612669675333709, 0.233194846820939)","(-0.03, -0.242, 0.078, 1.088)",0,1.0,"(-0.035, -0.453, 0.1, 1.748)",1.0,1.0,"(-0.044, -0.247, 0.135, 1.206)",...,0.078,1.088,-0.035,-0.453,0.100,1.748,-0.044,-0.247,0.135,1.206
1608,25,81,"(0.2055036802915485, 0.7357574690187201)","(0.199, 1.692, -0.156, -1.867)",1,1.0,"(0.233, 1.885, -0.193, -2.115)",0.0,1.0,"(0.271, 1.694, -0.236, -1.948)",...,-0.156,-1.867,0.233,1.885,-0.193,-2.115,0.271,1.694,-0.236,-1.948
2031,2,98,"(0.1955406148449595, 0.6245501454716056)","(0.05, 0.036, -0.049, -0.044)",0,1.0,"(0.05, -0.156, -0.05, 0.194)",0.0,1.0,"(0.047, -0.349, -0.046, 0.432)",...,-0.049,-0.044,0.050,-0.156,-0.050,0.194,0.047,-0.349,-0.046,0.432
1671,9,84,"(0.2011569802355368, 0.4470007793802826)","(0.054, 0.237, -0.007, -0.29)",0,1.0,"(0.059, 0.039, -0.012, 0.061)",1.0,1.0,"(0.059, 0.237, -0.011, -0.297)",...,-0.007,-0.290,0.059,0.039,-0.012,0.061,0.059,0.237,-0.011,-0.297
1792,26,89,"(0.1964578206703505, 1.7230322017264583)","(0.054, 0.769, -0.005, -0.369)",0,1.0,"(0.07, 0.583, -0.012, -0.278)",0.0,1.0,"(0.082, 0.397, -0.018, -0.188)",...,-0.005,-0.369,0.070,0.583,-0.012,-0.278,0.082,0.397,-0.018,-0.188


# Predict 

In [4]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [5]:
def reagroup(prediction_dataset):
    # agg_results = prediction_dataset[['episode'] + [
    #     f'rse_model_{i}' for i,_ in enumerate(models)
    # ]].groupby('episode').mean().reset_index()
    agg_results = prediction_dataset[['episode', 'step'] + [
        f'rse_model_{i}' for i,_ in enumerate(models)
    ]].copy()
    
    agg_results['best_model'] = agg_results.apply(lambda row: np.argmin(row[2:].values), axis=1)
    print(agg_results['best_model'].value_counts())

    prediction_dataset['best_model'] = prediction_dataset.apply(
        lambda row: agg_results.loc[(agg_results['episode']==row['episode']) & (agg_results['step']==row['step'])].best_model.values[0],
        axis=1
    )
    prediction_dataset['best_rse'] = prediction_dataset.apply(lambda row: row[f'rse_model_{row.best_model}'],axis=1)

    return prediction_dataset

In [6]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3']] = pre_df[['s0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3']]

    prediction_dataset = evaluate(models, pre_df)
    prediction_dataset = reagroup(prediction_dataset)
    final_predictions = evaluate(models, df)
    final_predictions['group'] = prediction_dataset['best_model']

    cols = [
        'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ]

    for c in cols:
        final_predictions[c] = final_predictions.apply(lambda x: x[f'{c}_model_{x.group}'], axis=1)

    return final_predictions[cols]

In [7]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

best_model
4    890
0    336
3    334
2    286
1    281
Name: count, dtype: int64


,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
728,"(-0.047, -0.208, 0.112, 1.293)",0.152,0.520804,0.003,0.039,0.023,0.087,0.549102,0.535644,0.556355,0.442115
1608,"(0.296, 1.632, -0.254, 0.586)",2.639,0.577372,0.025,0.062,0.018,2.534,0.572334,0.541298,0.544365,0.651493
2031,"(0.049, -0.357, -0.047, 0.621)",0.200,0.507628,0.002,0.008,0.001,0.189,0.548046,0.528024,0.503597,0.450843
1671,"(0.042, 0.237, -0.015, -0.167)",0.151,0.511632,0.017,0.000,0.004,0.130,0.563886,0.526057,0.510791,0.445794
1792,"(0.078, 0.393, -0.015, -0.134)",0.065,0.506221,0.004,0.004,0.003,0.054,0.550158,0.527040,0.508393,0.439292


In [8]:
prediction_dataset.describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
count,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.250589,0.512891,0.006946,0.017367,0.005508,0.220768,0.553269,0.530326,0.514407,0.453561
std,0.325422,0.014370,0.013888,0.032235,0.011323,0.301804,0.014666,0.007924,0.027154,0.025824
min,0.001000,0.502027,0.000000,0.000000,0.000000,0.000000,0.545935,0.526057,0.501199,0.434671
25%,0.083000,0.505885,0.001000,0.004000,0.001000,0.066000,0.546990,0.527040,0.503597,0.440318
50%,0.178000,0.508653,0.003000,0.009000,0.002000,0.156000,0.549102,0.528269,0.505995,0.448019
75%,0.290000,0.514344,0.006000,0.017000,0.005000,0.262000,0.552270,0.530236,0.513189,0.457089
max,4.357000,0.791670,0.308000,0.468000,0.285000,4.089000,0.871172,0.641101,1.184652,0.784547
